# 05 - model training

## 05.1 - mfcc + svm baseline

**purpose**: train a classical support vector machine on 240-dim mfcc feature vectors. this is the baseline required by the project definition, for comparison against the 1d cnn on mel-spectrograms (05.2).

**input**: `data/processed/X_train_mfcc.npy` (1932, 240), `y_train.npy`

**output**: `data/models/svm_best.pkl`, `data/models/mfcc_scaler.pkl`

**feature breakdown** (240 dims = 6 * 40 mfccs):

- mfcc[0..39] mean over time
- mfcc[0..39] std over time
- delta[0..39] mean over time
- delta[0..39] std over time
- delta2[0..39] mean over time
- delta2[0..39] std over time

**model**: rbf svm with gridsearch over c, gamma. class_weight='balanced' to handle the neutral/calm class imbalance in training set. probability=True so we can use predict_proba for the 08 ensemble.

all training logic lives in `src/models/svm_model.py`: `train_svm()` runs the gridsearch, `save_svm()` persists model + scaler, `predict_svm()` and `cpu_inference_latency_svm()` are used by 06/07/08.

In [1]:
import sys
import os
os.chdir("../../")
sys.path.append("../../")

**import svm training modules**. we bring in numpy, sklearn metrics, project settings, and the svm training and saving helpers.

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score
from src.config.config import settings
from src.utils.data_preprocessing import parse_filename
from src.models.svm_model import train_svm, save_svm, predict_svm

## Load training features (mfcc 240-dim)

In [3]:
X_train = np.load(settings.PROCESSED_DIR / "X_train_mfcc.npy")
y_train = np.load(settings.PROCESSED_DIR / "y_train.npy")

**verify training data shape**. quick sanity check on dimensions.

In [4]:
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

X_train: (1628, 240)
y_train: (1628,)


**check for nans and class balance**. we verify no missing values and print the per-class sample counts for reference.

In [5]:
print(f"Classes: {np.bincount(y_train)}")
assert X_train.shape[1] == settings.N_MFCC_VECTOR, (f"expected mfcc vector of {settings.N_MFCC_VECTOR} dims, got {X_train.shape[1]}")
assert not np.isnan(X_train).any(), "NaN in X_train"
print("X_train has 0 NaN values - OK")

Classes: [148 296 296 296 296 296]
X_train has 0 NaN values - OK


## Train SVM (gridsearchcv with 5-fold groupkfold by actor)

the function fits a standardscaler on the training set, then runs gridsearchcv over rbf / linear / polynomial kernels with multiple c and gamma values. cv uses GroupKFold(n_splits=5) with actor ids as groups, so each fold is speaker-disjoint (no actor appears in both train and val). scoring=accuracy. the best estimator is then refit on the full training set, and train accuracy is reported alongside the cv score as an overfitting indicator.

build actor groups for speaker-disjoint cv (groupkfold by actor)

In [6]:
train_actor_ids = []
for wav in sorted(settings.RAW_DIR.rglob("*.wav")):
    info = parse_filename(wav)
    if info is None:
        continue
    if info['emotion_id'] >= settings.NUM_CLASSES:
        continue
    if info['actor'] not in settings.TRAIN_ACTORS:
        continue
    train_actor_ids.append(info['actor'])

**verify actor id alignment**. we check that the actor id list has the same length as the training data and shows all 19 training actors.

In [7]:
train_actor_ids = np.array(train_actor_ids, dtype=np.int64)
assert len(train_actor_ids) == len(X_train), (
    f"actor_id list size {len(train_actor_ids)} != X_train size {len(X_train)}. "
    "the order may not match - check 03/04 iteration order."
)
print(f"train_actor_ids: {train_actor_ids.shape}, unique actors: {sorted(set(train_actor_ids.tolist()))}")

train_actor_ids: (1628,), unique actors: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


**run gridsearchcv with groupkfold**. we search over rbf, linear, and polynomial kernels with various C and gamma values. the best model (rbf, C=1, gamma=0.005) achieves 61% cross-validation accuracy on speaker-disjoint folds.

In [8]:
best_svm, scaler, cv_results = train_svm(X_train, y_train, groups=train_actor_ids)
print(f"Best CV score:   {cv_results['best_score'] * 100:.2f}%")
print(f"Best params:     {cv_results['best_params']}")
print(f"Best estimator:  kernel={best_svm.kernel}, C={best_svm.C}, gamma={best_svm.gamma}")
print(f"CV strategy:     GroupKFold(n_splits=5) by actor - speaker-disjoint folds")


Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best CV score:   61.07%
Best params:     {'C': 1, 'gamma': 0.005, 'kernel': 'rbf'}
Best estimator:  kernel=rbf, C=1, gamma=0.005
CV strategy:     GroupKFold(n_splits=5) by actor - speaker-disjoint folds


**compute training accuracy**. we run predictions on the full training set to check for overfitting. a large gap between train and cv accuracy would suggest the model memorized speakers.

In [9]:
train_preds, _ = predict_svm(best_svm, X_train, scaler)
svm_train_acc = accuracy_score(y_train, train_preds)


**print train-cv gap**. train accuracy (94.47%) is much higher than cv (61.07%), which is expected since the cv folds are on different speakers. this gap is the cross-speaker generalization cost.

In [10]:
print(f"Train accuracy:  {svm_train_acc * 100:.2f}%")
print(f"Gap (train - CV): {(svm_train_acc - cv_results['best_score']) * 100:+.2f}pp")

Train accuracy:  94.47%
Gap (train - CV): +33.41pp


## Save svm + scaler

In [11]:
svm_path, scaler_path = save_svm(best_svm, scaler, settings.MODELS_DIR)

**confirm saved model details**. we print the file paths, model dimensions, and class labels to confirm everything saved correctly.

In [12]:
print(f"SVM saved:    {svm_path}")
print(f"Scaler saved: {scaler_path} (+ scaler.pkl alias for back-compat)")
print(f"n_features_in_ = {best_svm.n_features_in_}")
print(f"classes_       = {best_svm.classes_}")

SVM saved:    E:\career\projects\lightweight-speech-emotion-recognition-on-open-datasets\data\models\svm_best.pkl
Scaler saved: E:\career\projects\lightweight-speech-emotion-recognition-on-open-datasets\data\models\mfcc_scaler.pkl (+ scaler.pkl alias for back-compat)
n_features_in_ = 240
classes_       = [0 1 2 3 4 5]
